# volcano-plots.ipynb
Volcano plots of q-value vs log odds ratio of variant_enrichment results.  
In R because EnhancedVolcano is better than any tooling in python.

In [ ]:
Sys.setenv(LANGUAGE = "en") # set language to "ja" if you prefer
suppressWarnings(library(EnhancedVolcano))
suppressWarnings(library(tidyverse))
suppressWarnings(library(ggplot2))
suppressWarnings(library(ggrepel))
suppressWarnings(library(extrafont))
suppressWarnings(library(svglite))
suppressWarnings(library(patchwork)) # combine plots

suppressMessages(extrafont::font_import(pattern="Arial",prompt=FALSE))
suppressMessages(extrafont::loadfonts())

getwd()
sessionInfo()

In [ ]:
# load data
p_res_path = "out/pathogenic_enrichment_results.tsv"
lp_res_path = "out/likely_pathogenic_enrichment_results.tsv"

read_results <- function(path){
    df <- read_tsv(path, col_names = FALSE) %>% suppressMessages
    lvl1 <- as.character(df[1, ])
    lvl2 <- as.character(df[2, ])
    new_names <- ifelse(is.na(lvl2) | lvl2 == "",
                    lvl1,
                    paste(lvl1, lvl2, sep = "_"))
    df <- df[-c(1, 2), ]
    names(df) <- new_names
    df <- type_convert(df)
    return(df)
}
p_df = read_results(p_res_path)
lp_df = read_results(lp_res_path)

In [ ]:
p_df %>% head

In [ ]:
plot_volcano <- function(df){
    # expects columns q-value, log_odds_ratio
    plt <- EnhancedVolcano(df,
        y='q-value',
        x='log_odds_ratio',
        lab=df$gene,
        title = NULL,
        subtitle = NULL,
        caption = NULL,
        axisLabSize = 14,
        labSize = 5,
        pCutoff = 0.05,
        FCcutoff = 0,
        drawConnectors = TRUE,
        widthConnectors = 0.5,
                           maxoverlapsConnectors = Inf,
        arrowheads = FALSE,
        col = rep("black",4),
    ) + 
        labs(y = expression(-Log[10](q))) +
        theme(legend.position = "none")
    return(plt)
}

save_plot <- function(plt,outfile,w=3.5,h=3.5){
    pngName = paste(outfile, ".png", sep="")
    svgName = paste(outfile, ".svg", sep = "")
    ggsave(path="out", device="png", filename=pngName, width=w, height=h, units='in')
    ggsave(path="out", device="svg", filename=svgName, width=w, height=h, units='in')
}

plot_volcano_wrapper <- function(df,comparison){
    # parse input data to send to the plotter function.
    # comparison should be 'ecDNA' or 'amplified'

    # format df
    df = df %>% 
        dplyr::select(gene,cancer_type,starts_with(comparison)) %>%
        rename_with(~ sub(paste0("^", comparison, "_"), "", .x),
                         starts_with(paste0(comparison, "_"))) %>%
    drop_na()

    #plot
    plt = plot_volcano(df)
    return(plt)
}

In [ ]:
options(repr.plot.width = 4, repr.plot.height = 4)
plt = plot_volcano_wrapper(p_df,'amplified')
save_plot(plt,"volcano_p_amp")
plt

In [ ]:
plt = plot_volcano_wrapper(p_df,'ecDNA')
save_plot(plt,"volcano_p_ec")
plt

In [ ]:
plt = plot_volcano_wrapper(lp_df,'amplified')
#save_plot(plt,"volcano_lp_amp")
plt

In [ ]:
plt = plot_volcano_wrapper(lp_df,'ecDNA')
save_plot(plt,"volcano_lp_ec")
plt

In [ ]:
?EnhancedVolcano

In [ ]:
osc_volcano_v <- function(stats_df,genes_of_interest){
    stats_df$highlight = ifelse(rownames(stats_df) %in% genes_of_interest,"notable","other")
    stats_df <- stats_df[order(stats_df$highlight=='other',decreasing=TRUE),]
    stats_df$color <- mapply(function(fc, padj, gene) {
      if (gene %in% genes_of_interest) {
        "blue4"  # custom color for notable
      } else if (fc < -1 & padj < 0.05) {
        "dodgerblue"   # upper-left
      } else if (fc > 1 & padj < 0.05) {
        "red"    # upper-right
      } else {
        "grey50" # everything else
      }
    }, stats_df$log2FoldChange, stats_df$padj, rownames(stats_df))

    plt <- EnhancedVolcano(stats_df,
                lab = rownames(stats_df),
                title = NULL,
                subtitle = NULL,
                caption = NULL,
                axisLabSize = 14,
                #x = 'logFC',
                #y = "adj.P.Val",
                #xlim = c(-2.75,2.75),
                x = 'log2FoldChange',
                y = 'padj',
                #ylim = c(0,4),
                pCutoff = 0.05,
                selectLab = genes_of_interest,
                #labSize = 4.21644413212,#3.37315530569,#
                #FCcutoff = 10,
                #vline = NULL, 
                #vlineType = "blank",
                legendPosition = "none",
                pointSize = c(ifelse(stats_df$highlight == "other", 1, 3)),
                colCustom = setNames(stats_df$color,rownames(stats_df)),
                drawConnectors = TRUE,
                maxoverlapsConnectors = Inf,
                lengthConnectors = unit(0, "npc"),
                colAlpha = c(ifelse(stats_df$highlight == "other", .6, .8)),
                ) %>% suppressWarnings
    options(repr.plot.width=7, repr.plot.height=7)
    return(plt + ylab(ylabel))
}